In [14]:
import sys
sys.path.insert(0, '.')
import pandas as pd
from src.parser import parse_deck_list
from src.api_client import enrich_deck
from src.deck import Deck
from src import calculator as calc
from src import monte_carlo as mc


In [15]:
# DECK_LIST = """
# Pokémon: 20
# 1 Alakazam MEP 9
# 2 Alakazam MEG 56
# 4 Kadabra MEG 55
# 4 Abra MEG 54
# 3 Dudunsparce PRE 80
# 2 Dunsparce JTG 120
# 1 Dunsparce TEF 128
# 1 Psyduck MEP 7
# 1 Fezandipiti ex SFA 38
# 1 Shaymin DRI 10

# Trainer: 33
# 1 Boss's Orders ASC 256
# 1 Boss's Orders PAL 265
# 1 Boss's Orders MEG 114
# 3 Rare Candy MEG 175
# 2 Battle Cage PFL 116
# 2 Battle Cage PFL 85
# 1 Sacred Ash POR 115
# 1 Eri TEF 146
# 3 Buddy-Buddy Poffin TEF 144
# 1 Buddy-Buddy Poffin ASC 184
# 2 Enhanced Hammer TWM 224
# 3 Poké Pad POR 81
# 1 Poké Pad POR 113
# 1 Lana's Aid TWM 155
# 4 Hilda WHT 84
# 4 Dawn PFL 87
# 1 Night Stretcher SFA 61
# 1 Wondrous Patch POR 117

# Energy: 7
# 1 Enriching Energy SSP 191
# 4 Telepathic Psychic Energy POR 88
# 2 Psychic Energy MEE 5
# """


In [16]:
DECK_LIST = """
Pokémon: 20
4 N's Zorua JTG 97
4 N's Zoroark ex JTG 98
2 N's Darumaka JTG 26
2 N's Darmanitan JTG 27
2 N's Zekrom ASC 155
1 N's Reshiram JTG 116
1 Budew ASC 16
1 Munkidori TWM 95
1 Yveltal MEG 88
1 Pecharunt ex SFA 39
1 Meowth ex POR 62

Trainer: 32
4 Lillie's Determination MEG 119
3 Cyrano SSP 170
3 Boss's Orders MEG 114
1 Janine's Secret Art PRE 112
1 Black Belt's Training JTG 143
4 Ultra Ball MEG 131
4 Buddy-Buddy Poffin TEF 144
3 Poké Pad ASC 198
3 N's PP Up JTG 153
2 Night Stretcher ASC 196
1 Unfair Stamp TWM 165
1 Binding Mochi PRE 95
2 N's Castle JTG 152

Energy: 8
8 Darkness Energy MEE 7
"""

In [17]:
# Optional: customize the target card and only the searches that can find it
TARGET_CARD_NAME = "N's Zorua"
TARGET_SEARCH_NAMES = [
    "Buddy-Buddy Poffin",
    "Poké Pad",
    "Ultra Ball"
    ]

# Monte Carlo settings
MC_SIMULATIONS = 100_000
MC_SEED = 42


In [18]:
print("Parsing deck list...")
parsed = parse_deck_list(DECK_LIST)
print(f"Found {len(parsed)} unique card entries. Looking up via TCGDex API (uses cache)...")
cards = enrich_deck(parsed, cache_path="card_cache.json")
deck = Deck(cards)
print(f"Total cards: {deck.total_cards} | Basic Pokemon: {deck.total_basics}")
unknown = [c for c in deck.cards if c.subcategory == 'unknown']
if unknown:
    print(f"WARNING: {len(unknown)} cards not found in API: {[c.name for c in unknown]}")
else:
    print("All cards classified successfully.")


Parsing deck list...
Found 25 unique card entries. Looking up via TCGDex API (uses cache)...
Total cards: 60 | Basic Pokemon: 14
All cards classified successfully.


In [19]:
print("=" * 50)
print("DECK BREAKDOWN")
print("=" * 50)

breakdown_data = {
    "Category": ["Pokémon", "", "Trainer", "", "", "", "Energy", ""],
    "Subcategory": ["Basic", "Other", "Item", "Supporter", "Stadium", "Tool", "Basic", "Special"],
    "#": [
        sum(c.quantity for c in deck.cards if c.subcategory == "basic"),
        sum(c.quantity for c in deck.cards if c.subcategory == "other"),
        deck.trainers_by_subtype.get("item", 0),
        deck.trainers_by_subtype.get("supporter", 0),
        deck.trainers_by_subtype.get("stadium", 0),
        deck.trainers_by_subtype.get("tool", 0),
        deck.energies_by_subtype.get("basic_energy", 0),
        deck.energies_by_subtype.get("special_energy", 0),
    ]
}
df_breakdown = pd.DataFrame(breakdown_data)
display(df_breakdown)
print(f"Total: {deck.total_cards} cards")


DECK BREAKDOWN


,Category,Subcategory,#
0,Pokémon,Basic,14
1,,Other,6
2,Trainer,Item,17
3,,Supporter,12
4,,Stadium,2
5,,Tool,1
6,Energy,Basic,8
7,,Special,0


Total: 60 cards


In [20]:
print("=" * 50)
print("OPENING HAND PROBABILITIES")
print("=" * 50)

N = deck.total_cards
b = deck.total_basics

opening_data = {
    "Event": ["Mulligan (no Basic)", "Starting with exactly 1 Basic", "Starting with 2 or more Basics"],
    "Probability": [
        f"{calc.mulligan_probability(N, b):.2%}",
        f"{calc.exactly_one_basic_probability(N, b):.2%}",
        f"{calc.two_or_more_basics_probability(N, b):.2%}",
    ]
}
display(pd.DataFrame(opening_data))


OPENING HAND PROBABILITIES


,Event,Probability
0,Mulligan (no Basic),13.86%
1,Starting with exactly 1 Basic,39.42%
2,Starting with 2 or more Basics,60.58%


In [21]:
print("=" * 50)
print("STARTER PROBABILITIES (per Basic Pokémon)")
print("=" * 50)

starter_data = []
for card in deck.basic_pokemon:
    possible = calc.possible_starter_probability(N, b, card.quantity)
    forced = calc.forced_starter_probability(N, b, card.quantity)
    starter_data.append({
        "Pokémon": f"{card.name} ({card.set_code} #{card.set_number})",
        "Copies": card.quantity,
        "Possible Starter": f"{possible:.2%}",
        "Forced Starter": f"{forced:.2%}",
    })
display(pd.DataFrame(starter_data))


STARTER PROBABILITIES (per Basic Pokémon)


,Pokémon,Copies,Possible Starter,Forced Starter
0,N's Zorua (JTG #97),4,46.38%,13.94%
1,N's Darumaka (JTG #26),2,25.71%,6.04%
2,N's Zekrom (ASC #155),2,25.71%,6.04%
3,N's Reshiram (JTG #116),1,13.54%,2.82%
4,Budew (ASC #16),1,13.54%,2.82%
5,Munkidori (TWM #95),1,13.54%,2.82%
6,Yveltal (MEG #88),1,13.54%,2.82%
7,Pecharunt ex (SFA #39),1,13.54%,2.82%
8,Meowth ex (POR #62),1,13.54%,2.82%


In [22]:
print("=" * 50)
print("PRIZE CARD PROBABILITIES")
print("=" * 50)

prize_data = []
for card in deck.cards:
    prob = calc.prize_probability(N, card.quantity)
    prize_data.append({
        "Card": card.name,
        "Copies": card.quantity,
        "P(≥1 Prized)": f"{prob:.2%}",
    })
df_prize = pd.DataFrame(prize_data).sort_values("P(≥1 Prized)", ascending=False)
display(df_prize.reset_index(drop=True))


PRIZE CARD PROBABILITIES


,Card,Copies,P(≥1 Prized)
0,Darkness Energy,8,59.33%
1,Lillie's Determination,4,35.15%
2,Buddy-Buddy Poffin,4,35.15%
3,Ultra Ball,4,35.15%
4,N's Zoroark ex,4,35.15%
5,N's Zorua,4,35.15%
6,Cyrano,3,27.52%
7,Boss's Orders,3,27.52%
8,N's PP Up,3,27.52%
9,Poké Pad,3,27.52%


In [23]:
print("=" * 50)
print("DRAW BY TURN")
print("=" * 50)

MAX_TURNS = 6
draw_data = []
for card in deck.cards:
    if card.quantity == 0:
        continue
    row = {"Card": card.name, "Copies": card.quantity}
    for t in range(1, MAX_TURNS + 1):
        row[f"Turn {t}"] = f"{calc.draw_by_turn_probability(N, card.quantity, t):.2%}"
    draw_data.append(row)
display(pd.DataFrame(draw_data))


DRAW BY TURN


,Card,Copies,Turn 1,Turn 2,Turn 3,Turn 4,Turn 5,Turn 6
0,N's Zorua,4,39.95%,44.48%,48.75%,52.77%,56.55%,60.10%
1,N's Zoroark ex,4,39.95%,44.48%,48.75%,52.77%,56.55%,60.10%
2,N's Darumaka,2,22.15%,25.08%,27.97%,30.79%,33.56%,36.27%
3,N's Darmanitan,2,22.15%,25.08%,27.97%,30.79%,33.56%,36.27%
4,N's Zekrom,2,22.15%,25.08%,27.97%,30.79%,33.56%,36.27%
5,N's Reshiram,1,11.67%,13.33%,15.00%,16.67%,18.33%,20.00%
6,Budew,1,11.67%,13.33%,15.00%,16.67%,18.33%,20.00%
7,Munkidori,1,11.67%,13.33%,15.00%,16.67%,18.33%,20.00%
8,Yveltal,1,11.67%,13.33%,15.00%,16.67%,18.33%,20.00%
9,Pecharunt ex,1,11.67%,13.33%,15.00%,16.67%,18.33%,20.00%


In [24]:
print("=" * 50)
print("SUPPORTER & DEAD HAND")
print("=" * 50)

s = deck.trainers_by_subtype.get("supporter", 0)
e = sum(deck.energies_by_subtype.values())

support_data = {
    "Statistic": [
        "Supporter in opening hand",
        "Dead Hand (0 Supporter + 0 Energy)",
    ],
    "Probability": [
        f"{calc.supporter_turn1_probability(N, s):.2%}",
        f"{calc.dead_hand_probability(N, s, e):.2%}",
    ]
}
display(pd.DataFrame(support_data))


SUPPORTER & DEAD HAND


,Statistic,Probability
0,Supporter in opening hand,80.94%
1,Dead Hand (0 Supporter + 0 Energy),4.83%


In [25]:
print("=" * 50)
print("SPECIFIC CARD + SEARCHERS")
print("=" * 50)

target_name = TARGET_CARD_NAME
target_copies = deck.quantity_of(target_name)
target_search_total = deck.quantity_of_names(TARGET_SEARCH_NAMES)

p_card = calc.specific_card_in_hand_probability(N, target_copies)
p_searcher = calc.searcher_probability(N, target_search_total)
p_either = calc.target_card_with_searches_probability(N, target_copies, target_search_total)

searcher_result = {
    "Statistic": [
        f"P({target_name} in opening hand)",
        f"P(Target search in hand) [{', '.join(TARGET_SEARCH_NAMES)}]",
        f"P({target_name} OR target search in hand)",
    ],
    "Probability": [
        f"{p_card:.2%}",
        f"{p_searcher:.2%}",
        f"{p_either:.2%}",
    ]
}
display(pd.DataFrame(searcher_result))


SPECIFIC CARD + SEARCHERS


,Statistic,Probability
0,P(N's Zorua in opening hand),39.95%
1,"P(Target search in hand) [Buddy-Buddy Poffin, ...",77.76%
2,P(N's Zorua OR target search in hand),88.25%


In [26]:
print("=" * 50)
print(f"MONTE CARLO VALIDATION (N={MC_SIMULATIONS:,} simulations)")
print("=" * 50)

print("Running simulation...")
sim = mc.simulate(deck, n=MC_SIMULATIONS, seed=MC_SEED)

# Compare key stats with theoretical values
comparison_data = [
    {
        "Statistic": "Mulligan rate",
        "Theoretical": f"{calc.mulligan_probability(N, b):.4f}",
        "Simulated": f"{sim['mulligan_rate']:.4f}",
        "Diff": f"{abs(sim['mulligan_rate'] - calc.mulligan_probability(N, b)):.4f}",
    },
    {
        "Statistic": "Supporter in hand",
        "Theoretical": f"{calc.supporter_turn1_probability(N, s):.4f}",
        "Simulated": f"{sim['supporter_in_hand']:.4f}",
        "Diff": f"{abs(sim['supporter_in_hand'] - calc.supporter_turn1_probability(N, s)):.4f}",
    },
    {
        "Statistic": "Dead hand",
        "Theoretical": f"{calc.dead_hand_probability(N, s, e):.4f}",
        "Simulated": f"{sim['dead_hand_rate']:.4f}",
        "Diff": f"{abs(sim['dead_hand_rate'] - calc.dead_hand_probability(N, s, e)):.4f}",
    },
]

# Add possible starters for each basic
for card in deck.basic_pokemon:
    theoretical = calc.possible_starter_probability(N, b, card.quantity)
    simulated = sim["possible_starters"].get(card.name, 0)
    comparison_data.append({
        "Statistic": f"Possible starter: {card.name}",
        "Theoretical": f"{theoretical:.4f}",
        "Simulated": f"{simulated:.4f}",
        "Diff": f"{abs(simulated - theoretical):.4f}",
    })

display(pd.DataFrame(comparison_data))
print("Monte Carlo validation complete.")


MONTE CARLO VALIDATION (N=100,000 simulations)
Running simulation...


,Statistic,Theoretical,Simulated,Diff
0,Mulligan rate,0.1386,0.1368,0.0018
1,Supporter in hand,0.8094,0.8062,0.0032
2,Dead hand,0.0483,0.0492,0.0009
3,Possible starter: N's Zorua,0.4638,0.4658,0.0021
4,Possible starter: N's Darumaka,0.2571,0.2570,0.0001
5,Possible starter: N's Zekrom,0.2571,0.2563,0.0008
6,Possible starter: N's Reshiram,0.1354,0.1371,0.0017
7,Possible starter: Budew,0.1354,0.1351,0.0004
8,Possible starter: Munkidori,0.1354,0.1347,0.0007
9,Possible starter: Yveltal,0.1354,0.1352,0.0002


Monte Carlo validation complete.
